### Dynamic Elasticity Verification — Clamped–Clamped Bar (ν = 0) with Damping

This notebook verifies transient linear elasticity on a bar with **both ends fixed**, **Poisson’s ratio ν = 0**, and **proportional damping** (e.g., $ \mathbf{C} = a\,\mathbf{M} $). The problem is advanced in time with a Newmark scheme and compared against the **analytical damped vibration** of the fundamental mode (reference frequency and exponential decay). We overlay **TRUST vs analytic** histories of mid-span displacement/velocity.

In [ ]:
from trustutils import run
import os
from math import sin, cos, exp, pi, sqrt

run.reset()
run.initBuildDirectory()

In [ ]:
L = 1
H = 1

E = 160e9
nu = 0.
rho = 7800
U0 = 0.01
omega1 = pi / L * (E / rho) ** 0.5
tmax = 2 * (2 * pi / omega1)
dt = 1e-6
alpha = 500
xi1 = alpha / (2 * omega1)
omegad1 = omega1 * (1 - xi1 **2) **0.5

has_ale = "TrioCFD_project_directory" in os.environ


run.addCaseFromTemplate("jdd.data", "results", {"E" : E, "nu" : nu, "L": L, "H": H, "rho": rho, "U0": U0, "tmax": tmax, "dt" : dt, "alpha": alpha})
if has_ale: run.addCaseFromTemplate("jdd_ale.data", "results", {"E" : E, "nu" : nu, "L": L, "H": H, "rho": rho, "U0": U0, "tmax": tmax, "dt" : dt, "alpha": alpha})
run.runCases()
run.tablePerf()

In [ ]:
from trustutils import plot
import numpy as np

fig = plot.Graph()
fig.addResidu(f"{run.BUILD_DIRECTORY}/results/jdd.dt_ev", label=" ")
if has_ale: fig.addResidu(f"{run.BUILD_DIRECTORY}/results/jdd_ale.dt_ev", label="ALE")
fig.scale("linear", "log")

t_interval = np.linspace(0, tmax, 1000)

fig = plot.Graph("u_x at the center")
fig.addPoint(f"{run.BUILD_DIRECTORY}/results/jdd_DX_MID.son", marker="-", label=f"TRUST-EL", compo=0, color="black")
if has_ale: fig.addPoint(f"{run.BUILD_DIRECTORY}/results/jdd_ale_DX_MID.son", marker="x", label=f"TRUST ALE", compo=0)
sol = [U0 * sin(pi * 0.5 / L) * exp(-xi1 * omega1 * t) * (cos(omegad1 * t) + xi1 * sin(omegad1 * t) / sqrt(1 - xi1 * xi1)) for t in t_interval]
fig.add(t_interval, sol, label=f"Exact solution", marker="--", color="red")
fig.label("Time [s]", "Displacement [m]")